# MegaVul Commit History Extraction

## 1. Setup

In [ ]:
# Prevents stale src loads
%load_ext autoreload
%autoreload 2

In [11]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve() / 'src'))

import pandas as pd

from vuln_commit_history import coming_runner, dbn_encode, git_ops, history, manifest, materialize, megavul_fetch
from vuln_commit_history.io import read_jsonl

# Setup
DATA_DIR = Path('..').resolve() / 'data'
MEGAVUL_JSON_PATH = DATA_DIR / 'megavul' / 'megavul.json'

# Anchor manifests
ANCHORS_PATH = DATA_DIR / 'manifests' / 'anchors.jsonl'

# Repositories
REPOSITORIES_DIR = DATA_DIR / 'repositories'

# History windows
TRANSITIONS_PATH = DATA_DIR / 'manifests' / 'transitions.jsonl'
WINDOW_SIZE = 10

# AST diffs
PAIRS_DIR = DATA_DIR / 'pairs'
COMING_OUTPUT_DIR = DATA_DIR / 'coming_output'
COMING_JAR = Path('..').resolve() / 'tools' / 'coming.jar'
JAVA_EXECUTABLE = r'C:\Program Files\Microsoft\jdk-17.0.20.101-hotspot\bin\java.exe'

# DBN encodings
LONG_CSV = DATA_DIR / 'datasets' / 'long_table.csv'
LONG_METADATA_JSON = DATA_DIR / 'datasets' / 'long_table_metadata.json'
WIDE_CSV = DATA_DIR / 'datasets' / 'wide_two_slice_table.csv'
MIN_SUPPORT = 5

print(DATA_DIR)

C:\Users\kvand\vuln-commit-history\data


In [3]:
megavul_fetch.ensure_megavul_json(MEGAVUL_JSON_PATH)
print('Found MegaVul dataset JSON at', MEGAVUL_JSON_PATH)

Found MegaVul dataset JSON at C:\Users\kvand\vuln-commit-history\data\megavul\megavul.json


## 2. Anchor manifests

In [4]:
anchors = manifest.write_anchor_manifest(MEGAVUL_JSON_PATH, ANCHORS_PATH)
projects = {a['project'] for a in anchors}
print(f'{len(anchors)} anchors across {len(projects)} projects')

902 anchors across 362 projects


## 3. Repositories

In [5]:
report = git_ops.ensure_all_repositories(anchors, REPOSITORIES_DIR, clone_missing=True)
print(f"cloned or present: {len(report['cloned_or_present'])}, failed: {len(report['failed'])}")
report['failed']

cloned or present: 311, failed: 72


[{'project': 'enonic/xp',
  'error': "Command failed (128): git clone --filter=blob:none --no-checkout https://github.com/enonic/xp.git C:\\Users\\kvand\\vuln-commit-history\\data\\repositories\\enonic_xp_0eee45a7c9\nCloning into 'C:\\Users\\kvand\\vuln-commit-history\\data\\repositories\\enonic_xp_0eee45a7c9'...\nfatal: unable to access 'https://github.com/enonic/xp.git/': Connection timed out after 318126 milliseconds"},
 {'project': 'esigate',
  'error': "Command failed (128): git clone --filter=blob:none --no-checkout https://github.com/esigate/esigate.git C:\\Users\\kvand\\vuln-commit-history\\data\\repositories\\esigate_1fb0b88c10\nCloning into 'C:\\Users\\kvand\\vuln-commit-history\\data\\repositories\\esigate_1fb0b88c10'...\nfatal: unable to access 'https://github.com/esigate/esigate.git/': Could not resolve host: github.com"},
 {'project': 'evandro-machado/Trabalho-Web2',
  'error': "Command failed (128): git clone --filter=blob:none --no-checkout https://github.com/evandro-ma

## 4. History windows

In [6]:
report = history.build_all_history_windows(
    ANCHORS_PATH, REPOSITORIES_DIR, TRANSITIONS_PATH, window_size=WINDOW_SIZE, clone_missing=True
)
print(report['anchors_processed'], 'anchors processed;', len(report['anchors_failed']), 'failed')
print(report['transitions'], 'total transitions')

899 anchors processed; 3 failed
7357 total transitions


In [7]:
transitions = read_jsonl(TRANSITIONS_PATH)
preview = pd.DataFrame(transitions)
if not preview.empty:
    one_anchor = preview['anchor_id'].iloc[0]
    display_cols = ['sample_id', 'SLICE_OFFSET', 'TRANSITION_LABEL', 'source_commit', 'target_commit', 'target_timestamp']
    preview[preview['anchor_id'] == one_anchor][display_cols].sort_values('SLICE_OFFSET')

## 5. AST diffs

In [8]:
materialize_report = materialize.materialize_transitions(TRANSITIONS_PATH, REPOSITORIES_DIR, PAIRS_DIR)
print(len(materialize_report['materialized']), 'materialized;', len(materialize_report['failed']), 'failed')

7028 materialized; 329 failed


In [12]:
coming_report = coming_runner.run_coming(
    TRANSITIONS_PATH, PAIRS_DIR, COMING_OUTPUT_DIR, COMING_JAR, workers=4, java=JAVA_EXECUTABLE
)
print(len(coming_report['succeeded']), 'succeeded;', len(coming_report['failed']), 'failed')

7028 succeeded; 329 failed


## 6. DBN encodings

In [13]:
long_df = dbn_encode.build_long_table(
    TRANSITIONS_PATH, COMING_OUTPUT_DIR, LONG_CSV, LONG_METADATA_JSON, min_support=MIN_SUPPORT
)
feature_columns = [c for c in long_df.columns if c.startswith('CT__')]
print(len(long_df), 'transitions;', len(feature_columns), 'retained change-type features')

7028 transitions; 467 retained change-type features


In [14]:
wide_df = dbn_encode.build_wide_two_slice_table(long_df, feature_columns)
wide_df.to_csv(WIDE_CSV, index=False)
print(len(wide_df), 'two-slice training instances across', wide_df['anchor_id'].nunique(), 'anchors')

6168 two-slice training instances across 852 anchors


## 7. Sanity checks

In [15]:
long_df = pd.read_csv(LONG_CSV)
wide_df = pd.read_csv(WIDE_CSV)

In [16]:
# Window-length distribution
long_df.groupby('anchor_id').size().describe()

count    860.000000
mean       8.172093
std        3.494638
min        1.000000
25%        5.000000
50%       11.000000
75%       11.000000
max       11.000000
dtype: float64

In [17]:
# Label balance
long_df['TRANSITION_LABEL'].value_counts()

TRANSITION_LABEL
OTHER    6168
FIX       860
Name: count, dtype: int64

In [18]:
# Feature coverage
feature_columns = [c for c in long_df.columns if c.startswith('CT__')]
long_df[feature_columns].sum().sort_values(ascending=False).head(20)

CT__INSERT_NODE_METHOD_CLASS               1115
CT__INSERT_NODE_IF_BLOCK                   1107
CT__INSERT_NODE_LOCALVARIABLE_BLOCK         922
CT__INSERT_NODE_INVOCATION_BLOCK            860
CT__INSERT_NODE_FIELD_CLASS                 706
CT__MOVE_TREE_INVOCATION_BLOCK              653
CT__MOVE_TREE_VARIABLEREAD_INVOCATION       581
CT__DELETE_NODE_LOCALVARIABLE_BLOCK         501
CT__INSERT_NODE_VARIABLEREAD_INVOCATION     496
CT__MOVE_TREE_IF_BLOCK                      447
CT__INSERT_NODE_ASSIGNMENT_BLOCK            440
CT__DELETE_NODE_INVOCATION_BLOCK            435
CT__MOVE_TREE_INVOCATION_INVOCATION         432
CT__MOVE_TREE_LOCALVARIABLE_BLOCK           432
CT__UPDATE_NODE_INVOCATION_BLOCK            416
CT__DELETE_NODE_IF_BLOCK                    415
CT__UPDATE_NODE_METHOD_CLASS                328
CT__UPDATE_NODE_LOCALVARIABLE_BLOCK         322
CT__UPDATE_NODE_INVOCATION_INVOCATION       317
CT__DELETE_NODE_METHOD_CLASS                317
dtype: int64